# Simple RAG



In [ ]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

import os, tiktoken
import pandas as pd

from tqdm import tqdm
import numpy as np

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import gradio as gr

load_dotenv(override=True)

URL_Reactome_Pathways = "https://download.reactome.org/95/ReactomePathways.txt"
URL_Reactome_Summary  = "https://reactome.org/download/current/pathway2summation.txt"

MODEL = "gpt-4.1-nano"

DB_NAME = "reactome_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

root_data = '../data'
root_reactome =  os.path.join(root_data, 'reactome')
root_vector_store = os.path.join(root_data, 'vector_store')
root_tcga =  os.path.join(root_data, 'tcga')

openai = OpenAI()

In [ ]:
os.listdir(root_reactome)

### Get Reactome

In [ ]:
def get_reactome_pathways(which:str="pathways", force:bool=False) -> pd.DataFrame:

    if which == "pathways":
        url = URL_Reactome_Pathways
        fname = "reactome_pathways.tsv"
        names=["reactome_id", "pathway", "species"]
    else:
        url = URL_Reactome_Summary
        fname = "reactome_summary.tsv"
        names=["reactome_id", "name", "summary", 'any2']

    filename = os.path.join(root_reactome, fname)

    if os.path.exists(filename) and not force:
        df = pd.read_csv(filename, sep="\t")
        print(f"File {filename} already exists: {df.shape}.")
        return df

    try:
        df = pd.read_csv(url,  sep="\t", header=None, names=names)
        df = df.drop(columns=['any2'], errors='ignore')
        df = df.iloc[1:]

    except Exception as e:
        print(f"Error fetching data from Reactome: {e}")
        return pd.DataFrame()
    
    df.to_csv(filename, sep="\t", index=False)

    return df

In [ ]:
df = get_reactome_pathways(which="pathways", force=False)
print(len(df))

df.head(3)

In [ ]:
dfs = get_reactome_pathways(which="summary", force=False)
print(len(dfs))

df.head(3)

In [ ]:
dfa = df[df.species == "Homo sapiens"]
print(len(dfa))

In [ ]:
dfs2 = dfs[dfs.reactome_id.isin(dfa.reactome_id)]
dfs2 = dfs2.drop_duplicates('reactome_id')
print(len(dfs2))

dfs2.tail(3)

In [ ]:
text = ''

for i, row in dfs2.iterrows():
    text += f"{row['reactome_id']}  | {row['name']} + | + {row['summary']}\n"

print('----------end----------\n', i)

### Tokens

In [ ]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(text)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

### Documents

In [ ]:
fname = "reactome_summary.tsv"
filename = os.path.join(root_reactome, fname)

if os.path.exists(filename):
    loader = TextLoader(filename, encoding="utf-8")
    documents = loader.load()
else:
    print("Nothing found")
    documents = []

print(documents[0].page_content)

In [ ]:
print(documents[0].metadata)

### Divide into chunks using the RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=25)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

### Pick an embedding model

In [ ]:
dir_VS = os.path.join(root_vector_store, DB_NAME)
DB_NAME, dir_VS, os.path.exists(dir_VS)

In [ ]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(dir_VS):
    # if the database already exists, delete it and create a new one
    # Chroma(persist_directory=dir_VS, embedding_function=embeddings).delete_collection()
    print(f"Database {dir_VS} already exists. Skipping creation.")
    vectorstore = Chroma(persist_directory=dir_VS, embedding_function=embeddings)
else:
    # langchain's Chroma wrapper for chromadb
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=dir_VS)

print(f"Vectorstore created with {vectorstore._collection.count()} documents")

### Let's investigate the vectors

In [ ]:

collection = vectorstore._collection
count = collection.count()

lista = collection.get(limit=1, include=["embeddings"])['embeddings']
print(type(lista))
sample_embedding = np.array(lista)[0]

dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

#### Retriever

In [ ]:
retriever = vectorstore.as_retriever()

### Vector similarity

In [ ]:
question = "Innate Immune System"

retriever.invoke(question)

### RAG

In [ ]:
llm = ChatOpenAI(temperature=0, model=MODEL)

SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, a Reactome assistant.
You are chatting with a user about Reactome's pathways.
If relevant, use the given context to answer any scientific question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history: list) -> str:
    # Retrieve relevant documents from the vector store
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Format the system prompt with the retrieved context
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

    # Invoke the LLM with the system prompt and the user's question
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return str(response.content)



In [ ]:
question = "Does a bacteria invasion elicits an innate immune response? Which are the main pathways involved?"
answer_question(question, [])

In [ ]:
gr_question = gr.ChatInterface(
    fn=answer_question,
    chatbot=gr.Chatbot(label="messages"),
)

gr_question.launch()